# EXP-04 (`CLOUD-LANE-01`): Paired A/B `vLLM 0.15+` + `CowrieKVConnectorV1` Benchmark & `SupportsHMA` Proof

**Goal:** Run a **paired A/B benchmark** on the **exact same GPU, model, prompts, `max_model_len`, `gpu_memory_utilization`, and `seed`** across a **full two-process OS restart** (`Phase 1: Cold Store` -> complete VRAM teardown -> `Phase 2: Post-Restart Restore`):

- **ARM A (`arm_a_without_cowrie`, control):** Plain vLLM, **NO `CowrieKVConnectorV1`**, native prefix caching **ON** (`enable_prefix_caching=True`). Measures baseline cold prefill and post-restart prefill relying solely on vLLM's native in-GPU prefix cache (honestly recording 0% cross-restart cache retention when volatile VRAM is wiped by OS process exit).
- **ARM B (`arm_b_with_cowrie`, treatment):** vLLM + **`CowrieKVConnectorV1`** + native prefix caching **ON**. Saves multi-layer KV blocks to `0600` HMAC-SHA256 + 4 MiB Merkle `.state.cowrie` shards on disk in Process 1 and restores them into fresh Process 2.

Both arms record the **full metric set and connector counters** into **one unified receipt** with an explicit `ab` section and `delta_table`, proving:
1. `vllm.general_plugins` / `KVConnectorFactory` registers `CowrieKVConnectorV1`.
2. **`SupportsHMA` is genuinely `True` at runtime** (statically on the factory-resolved class via real `vllm...SupportsHMA` and dynamically on live Scheduler/Worker instances).
3. **100% exact greedy token parity** holds within ARM A, within ARM B, and across ARM A vs ARM B (hard gate: no receipt if parity fails in either arm).
4. Direct **Cowrie vs native prefix cache** post-restart TTFT, decode tok/s, new-suffix reuse, KV cache hit rate, and connector counters — failing loudly if the connector is silently bypassed.

### Hardware Profiles, Final Colab Roster & Credit Pool
- **Credit pool:** Google AI Ultra — **Google Colab Compute Units** (`2,000 CU/mo`, 90-day rollover). **Zero paid GCP spend, zero external accounts, no gated HF token required.**
- **Primary GPU:** **NVIDIA H100 80GB HBM3 (`sm_90`)** (~18 CU/hr).
- **Blackwell variant (supported):** **NVIDIA RTX PRO 6000 Blackwell 96GB (`sm_120`, G4)** (~8.9 CU/hr). `CowrieKVConnectorV1` natively supports FlashAttention `(2, blocks, ...)`, FlashInfer `(blocks, 2, ...)`, and MLA KV layouts.
- **Final Colab single-GPU profiles (all quantized; zero BF16 Colab profiles):**
  1. **`qwen3.8-27b-fp8`** *(DEFAULT)* — `Qwen/Qwen3.8-27B-FP8` (`27,781,427,952` params, official `F8_E4M3`, `25.87 GiB` raw weights; hits explicit fail-closed hybrid linear-attention gate in `layout_ver=3`).
  2. **`qwen3.6-27b-fp8`** — `Qwen/Qwen3.6-27B-FP8` (`27.8B` params, official `F8_E4M3`, `25.89 GiB` raw weights).
  3. **`inkling-2-mini`** — `thinkingmachines/Inkling-2-Mini` (`26,388,646,440` params, already-quantized mixed `U8`/`F8_E4M3` + `1.9B` `BF16`, `<=26.35 GiB` raw weights; do not convert).
  4. **`glm-4.7-flash-awq`** — `cyankiwi/GLM-4.7-Flash-AWQ-4bit` (`~31B` params, **community 4-bit AWQ** via vLLM AWQ path, `17.23 GiB` est. raw weights; official `zai-org/GLM-4.7-Flash` is BF16-only at `~62 GB` and excluded).
- **Post-EIN multi-GPU targets (NOT Colab profiles — see `docs/CLOUD-LANE-01.md`):** `zai-org/GLM-5.3-Flash` (`321,323,031,390` params, 8xH100 post-EIN target only) and `deepseek-ai/DeepSeek-V4.1-Flash-1017` (`763,205,315,794` params, multi-node / 8xH200 post-EIN target only).

> **Instructions:** Select an H100 80GB (or RTX PRO 6000 Blackwell 96GB) runtime in Colab Ultra and click **Runtime -> Run all**. No manual edits are required.

In [ ]:
import json
import os
import secrets
from pathlib import Path

# ==============================================================================
# Experiment Configuration (zero manual edits required for H100 80GB default)
# All 4 profiles run the exact same benchmark settings for apples-to-apples comparison.
# ==============================================================================
MODEL_PROFILE = os.environ.get("EXP04_MODEL_PROFILE", "qwen3.8-27b-fp8")

BENCH_MAX_MODEL_LEN = 8192
BENCH_GPU_MEMORY_UTILIZATION = 0.90
BENCH_SEED = int(os.environ.get("EXP04_SEED", "42"))

RAW_MODEL_SPECS = {
    # 1. DEFAULT — Official FP8 build of Qwen3.8-27B
    "qwen3.8-27b-fp8": {
        "model_id": "Qwen/Qwen3.8-27B-FP8",
        "provenance": "official-quantized",
        "params": 27_781_427_952,
        "active_params": 27_781_427_952,
        "quantized_params": 27_781_427_952,
        "quantized_bytes_per_param": 1.0,
        "unquantized_params": 0,
        "unquantized_bytes_per_param": 2.0,
        "weight_dtype": "F8_E4M3",
        "requires_fp8_hw": True,
        "vllm_dtype": "auto",
        "quantization": None,
        "trust_remote_code": True,
        "max_model_len": BENCH_MAX_MODEL_LEN,
        "gpu_memory_utilization": BENCH_GPU_MEMORY_UTILIZATION,
        "estimated_runtime_overhead_gib": 5.0,
        "min_kv_headroom_gib": 8.0,
        "notes": "DEFAULT single-H100 profile; official F8_E4M3 checkpoint (~25.9 GiB raw weights).",
    },
    # 2. Official FP8 build of Qwen3.6-27B (27.8B)
    "qwen3.6-27b-fp8": {
        "model_id": "Qwen/Qwen3.6-27B-FP8",
        "provenance": "official-quantized",
        "params": 27_800_000_000,
        "active_params": 27_800_000_000,
        "quantized_params": 27_800_000_000,
        "quantized_bytes_per_param": 1.0,
        "unquantized_params": 0,
        "unquantized_bytes_per_param": 2.0,
        "weight_dtype": "F8_E4M3",
        "requires_fp8_hw": True,
        "vllm_dtype": "auto",
        "quantization": None,
        "trust_remote_code": True,
        "max_model_len": BENCH_MAX_MODEL_LEN,
        "gpu_memory_utilization": BENCH_GPU_MEMORY_UTILIZATION,
        "estimated_runtime_overhead_gib": 5.0,
        "min_kv_headroom_gib": 8.0,
        "notes": "Official F8_E4M3 build of the 27.8B Qwen3.6 model.",
    },
    # 3. Already-quantized mixed U8/F8_E4M3 (+ 1.9B BF16) — do NOT convert
    "inkling-2-mini": {
        "model_id": "thinkingmachines/Inkling-2-Mini",
        "provenance": "official-quantized",
        "params": 26_388_646_440,
        "active_params": 3_000_000_000,
        "quantized_params": 24_488_646_440,
        "quantized_bytes_per_param": 1.0,  # conservative 1 B/param bound for mixed U8/F8_E4M3
        "unquantized_params": 1_900_000_000,
        "unquantized_bytes_per_param": 2.0,  # 1.9B params retained in BF16
        "weight_dtype": "MIXED_U8_F8_E4M3_1.9B_BF16",
        "requires_fp8_hw": True,
        "vllm_dtype": "auto",
        "quantization": None,  # already quantized on HF; load natively without conversion
        "trust_remote_code": True,
        "max_model_len": BENCH_MAX_MODEL_LEN,
        "gpu_memory_utilization": BENCH_GPU_MEMORY_UTILIZATION,
        "estimated_runtime_overhead_gib": 4.0,
        "min_kv_headroom_gib": 8.0,
        "notes": "Already quantized (mixed U8/F8_E4M3 with only 1.9B BF16); do not convert.",
    },
    # 4. Community 4-bit AWQ quantization of GLM-4.7-Flash (official zai-org/GLM-4.7-Flash is BF16-only ~62 GiB)
    "glm-4.7-flash-awq": {
        "model_id": "cyankiwi/GLM-4.7-Flash-AWQ-4bit",
        "provenance": "community-quantized (not official; official zai-org/GLM-4.7-Flash is BF16-only ~62 GiB)",
        "params": 31_000_000_000,
        "active_params": 3_000_000_000,
        "quantized_params": 29_000_000_000,
        "quantized_bytes_per_param": 0.5,  # 4-bit AWQ weights
        "unquantized_params": 2_000_000_000,
        "unquantized_bytes_per_param": 2.0,  # unquantized embed/lm_head/norms
        "weight_dtype": "AWQ_INT4",
        "requires_fp8_hw": False,
        "vllm_dtype": "auto",
        "quantization": None,  # vLLM auto-selects AWQ/Marlin path from checkpoint quantization_config
        "trust_remote_code": True,
        "max_model_len": BENCH_MAX_MODEL_LEN,
        "gpu_memory_utilization": BENCH_GPU_MEMORY_UTILIZATION,
        "estimated_runtime_overhead_gib": 5.5,  # AWQ group scales/zeros + CUDA/vLLM runtime
        "min_kv_headroom_gib": 8.0,
        "notes": "Community 4-bit AWQ quantization (vLLM AWQ path); not official zai-org build.",
    },
}


def compute_memory_footprint(spec: dict) -> dict:
    """Compute raw weight footprint from parameter counts and bytes/param, plus GPU fit threshold."""
    assert spec["quantized_params"] + spec["unquantized_params"] == spec["params"], (
        f"Param split mismatch for {spec['model_id']}"
    )
    assert spec["quantized_params"] > 0 and not spec["weight_dtype"].upper().startswith("BF16"), (
        f"Runnable Colab profile {spec['model_id']} MUST be quantized (BF16-only profiles forbidden)."
    )
    q_bytes = int(round(spec["quantized_params"] * spec["quantized_bytes_per_param"]))
    uq_bytes = int(round(spec["unquantized_params"] * spec["unquantized_bytes_per_param"]))
    raw_bytes = q_bytes + uq_bytes
    raw_weight_gib = round(raw_bytes / (1024 ** 3), 2)
    raw_weight_gb = round(raw_bytes / 1e9, 2)
    min_required_vram_gib = round(
        raw_weight_gib + spec["estimated_runtime_overhead_gib"] + spec["min_kv_headroom_gib"],
        2,
    )
    return {
        **spec,
        "raw_weight_bytes": raw_bytes,
        "computed_weight_gib": raw_weight_gib,
        "computed_weight_gb": raw_weight_gb,
        "min_required_vram_gib": min_required_vram_gib,
    }


MODEL_PROFILES = {k: compute_memory_footprint(v) for k, v in RAW_MODEL_SPECS.items()}

assert MODEL_PROFILE in MODEL_PROFILES, (
    f"Unknown EXP04_MODEL_PROFILE={MODEL_PROFILE!r}; valid Colab profiles: {sorted(MODEL_PROFILES)}"
)
PROFILE = MODEL_PROFILES[MODEL_PROFILE]

COWRIE_REPO_URL = os.environ.get("COWRIE_REPO_URL", "https://github.com/Neumenon/cowrie.git")
COWRIE_GIT_REF = os.environ.get("COWRIE_GIT_REF", "feat/cloud-lane-01")
STATE_DIR = Path(os.environ.get("EXP04_STATE_DIR", "/tmp/cowrie_exp04_kv"))
RECEIPT_PATH = Path(os.environ.get("EXP04_RECEIPT_PATH", "/tmp/cowrie_exp04_receipt.json"))

# Identical benchmark workload across all profiles for directly comparable numbers.
PROMPT_ROWS = int(os.environ.get("EXP04_PROMPT_ROWS", "120"))
DECODE_MAX_TOKENS = int(os.environ.get("EXP04_DECODE_TOKENS", "64"))
# Eager mode keeps 2-process startup fast & uniform across Hopper sm_90 and Blackwell sm_120.
# Set EXP04_ENFORCE_EAGER=0 to benchmark with torch.compile / CUDA graphs.
ENFORCE_EAGER = os.environ.get("EXP04_ENFORCE_EAGER", "1") == "1"

# Raise host-RAM staging cap above the 512 MiB default so multi-thousand-token shards never silently skip.
COWRIE_MAX_INFLIGHT_BYTES = 4 * 1024 * 1024 * 1024  # 4 GiB host-RAM staging cap
COWRIE_MAX_CACHE_BYTES = 32 * 1024 * 1024 * 1024    # 32 GiB local disk cache cap

# Ephemeral per-run 256-bit HMAC key (no external secrets required).
COWRIE_HMAC_KEY_HEX = os.environ.get("COWRIE_HMAC_KEY", secrets.token_hex(32))

# Set True for unattended runs to automatically disconnect Colab runtime at the end.
AUTO_UNASSIGN_COLAB_RUNTIME = os.environ.get("EXP04_AUTO_UNASSIGN", "0") == "1"

CONFIG = {
    "experiment": "EXP-04",
    "lane": "CLOUD-LANE-01",
    "model_profile": MODEL_PROFILE,
    "model_id": PROFILE["model_id"],
    "provenance": PROFILE["provenance"],
    "params": PROFILE["params"],
    "active_params": PROFILE["active_params"],
    "weight_dtype": PROFILE["weight_dtype"],
    "computed_weight_gib": PROFILE["computed_weight_gib"],
    "computed_weight_gb": PROFILE["computed_weight_gb"],
    "estimated_runtime_overhead_gib": PROFILE["estimated_runtime_overhead_gib"],
    "min_kv_headroom_gib": PROFILE["min_kv_headroom_gib"],
    "min_required_vram_gib": PROFILE["min_required_vram_gib"],
    "dtype": PROFILE["vllm_dtype"],
    "quantization": PROFILE["quantization"],
    "trust_remote_code": PROFILE["trust_remote_code"],
    "max_model_len": PROFILE["max_model_len"],
    "gpu_memory_utilization": PROFILE["gpu_memory_utilization"],
    "seed": BENCH_SEED,
    "prompt_rows": PROMPT_ROWS,
    "decode_max_tokens": DECODE_MAX_TOKENS,
    "enforce_eager": ENFORCE_EAGER,
    "cowrie_max_inflight_bytes": COWRIE_MAX_INFLIGHT_BYTES,
    "cowrie_max_cache_bytes": COWRIE_MAX_CACHE_BYTES,
    "state_dir": str(STATE_DIR),
    "receipt_path": str(RECEIPT_PATH),
    "cowrie_repo_url": COWRIE_REPO_URL,
    "cowrie_git_ref": COWRIE_GIT_REF,
    "auto_unassign_colab_runtime": AUTO_UNASSIGN_COLAB_RUNTIME,
}

print("Final quantized Colab single-GPU profiles (computed footprints before load):")
for name, p in MODEL_PROFILES.items():
    marker = " <-- ACTIVE" if name == MODEL_PROFILE else ""
    print(
        f"  - {name}: {p['model_id']} [{p['provenance']}] | params={p['params']:,} "
        f"(active={p['active_params']:,}) | {p['weight_dtype']} -> "
        f"{p['computed_weight_gib']:.2f} GiB raw weights ({p['computed_weight_gb']:.2f} GB) | "
        f"min_required_vram={p['min_required_vram_gib']:.2f} GiB{marker}"
    )
print()
print(json.dumps(CONFIG, indent=2))

In [ ]:
def make_corpus_prefix(corpus_id: str, rows: int) -> str:
    """Build a deterministic, distinct long structured prefix so prompts do not collide in VRAM."""
    lines = [
        f"corpus={corpus_id} node=gpu-{i:03d} rack=r{i % 8} "
        f"temp_c={51 + ((i * 7 + len(corpus_id)) % 23)} "
        f"util_pct={72 + ((i * 5) % 27)} mem_gib={61 + (i % 18)} status=healthy"
        for i in range(rows)
    ]
    return (
        f"You are an infrastructure diagnostics assistant reviewing verified telemetry batch {corpus_id}.\n"
        + "\n".join(lines)
        + "\n\n"
    )


def build_benchmark_prompts(rows: int) -> dict[str, str]:
    """Return three disjoint prefix corpora for TTFT, Decode+Parity, and New-Suffix reuse."""
    prefix_ttft = make_corpus_prefix("EXP04_TTFT_ALPHA", rows)
    prefix_decode = make_corpus_prefix("EXP04_DECODE_BETA", rows)
    prefix_suffix = make_corpus_prefix("EXP04_SUFFIX_GAMMA", rows)
    return {
        "prompt_ttft": (
            prefix_ttft
            + "Question: Which node index is the final entry in batch EXP04_TTFT_ALPHA?\nAnswer:"
        ),
        "prompt_decode": (
            prefix_decode
            + "Question: Summarize the cluster telemetry trends across racks r0 through r7, "
            "noting temperature spread, utilization range, and the final node index.\nAnswer:"
        ),
        # Phase 1 seeds the exact block-aligned prefix of prefix_suffix;
        # Phase 2 queries prefix_suffix with a brand-new suffix question.
        "prompt_prefix_seed": prefix_suffix,
        "prompt_new_suffix": (
            prefix_suffix
            + "Question: Identify the highest rack index and summarize overall node health.\nAnswer:"
        ),
    }


def check_model_fits_gpu(profile: dict, total_vram_gib: float, free_vram_gib: float, compute_cap: str) -> dict:
    """Verify that the active model profile fits the attached GPU before downloading or loading weights."""
    usable_vram_gib = round(total_vram_gib * float(profile["gpu_memory_utilization"]), 2)
    loaded_est_gib = round(profile["computed_weight_gib"] + profile["estimated_runtime_overhead_gib"], 2)
    projected_kv_headroom_gib = round(usable_vram_gib - loaded_est_gib, 2)
    cap_parts = tuple(int(p) for p in compute_cap.split(".")[:2])

    fit_report = {
        "model_id": profile["model_id"],
        "provenance": profile["provenance"],
        "params": profile["params"],
        "active_params": profile["active_params"],
        "weight_dtype": profile["weight_dtype"],
        "computed_weight_gib": profile["computed_weight_gib"],
        "computed_weight_gb": profile["computed_weight_gb"],
        "estimated_runtime_overhead_gib": profile["estimated_runtime_overhead_gib"],
        "estimated_loaded_gib": loaded_est_gib,
        "min_kv_headroom_gib": profile["min_kv_headroom_gib"],
        "min_required_vram_gib": profile["min_required_vram_gib"],
        "gpu_total_vram_gib": total_vram_gib,
        "gpu_free_vram_gib": free_vram_gib,
        "gpu_usable_vram_gib": usable_vram_gib,
        "projected_kv_headroom_gib": projected_kv_headroom_gib,
        "compute_capability": compute_cap,
        "fits": False,
    }

    errors: list[str] = []
    if profile.get("requires_fp8_hw") and cap_parts < (8, 9):
        errors.append(
            f"{profile['model_id']} uses native {profile['weight_dtype']} weights requiring sm_89+ "
            f"(Hopper/Ada/Blackwell), but attached GPU is sm_{compute_cap.replace('.', '')}."
        )
    if free_vram_gib < profile["min_required_vram_gib"]:
        errors.append(
            f"Free VRAM ({free_vram_gib:.2f} GiB) < required ({profile['min_required_vram_gib']:.2f} GiB = "
            f"{profile['computed_weight_gib']:.2f} GiB weights + {profile['estimated_runtime_overhead_gib']:.2f} GiB overhead + "
            f"{profile['min_kv_headroom_gib']:.2f} GiB min KV headroom)."
        )
    if usable_vram_gib < profile["min_required_vram_gib"]:
        errors.append(
            f"Usable VRAM at gpu_memory_utilization={profile['gpu_memory_utilization']} ({usable_vram_gib:.2f} GiB) "
            f"< required ({profile['min_required_vram_gib']:.2f} GiB)."
        )
    if projected_kv_headroom_gib < profile["min_kv_headroom_gib"]:
        errors.append(
            f"Projected KV headroom ({projected_kv_headroom_gib:.2f} GiB) < minimum ({profile['min_kv_headroom_gib']:.2f} GiB)."
        )

    if errors:
        raise RuntimeError(
            f"GPU MEMORY FIT CHECK FAILED for {profile['model_id']}:\n  - " + "\n  - ".join(errors)
        )
    fit_report["fits"] = True
    return fit_report


def _check_greedy_parity(label: str, ref_toks: list[int], got_toks: list[int], errors: list[str]) -> None:
    if not ref_toks or ref_toks != got_toks:
        diff_idx = next(
            (i for i, (a, b) in enumerate(zip(ref_toks, got_toks)) if a != b),
            min(len(ref_toks), len(got_toks)),
        )
        errors.append(
            f"{label} FAILED at token {diff_idx}: "
            f"ref[{diff_idx}:{diff_idx+4}]={ref_toks[diff_idx:diff_idx+4]} vs "
            f"got[{diff_idx}:{diff_idx+4}]={got_toks[diff_idx:diff_idx+4]}"
        )


def assert_control_arm_valid(store_res: dict, restore_res: dict) -> None:
    """Fail loudly if ARM A (control without Cowrie) accidentally invoked Cowrie or failed greedy parity."""
    errors: list[str] = []
    st = store_res.get("connector_stats", {}).get("totals", {})
    rt = restore_res.get("connector_stats", {}).get("totals", {})

    if st.get("instance_count", 0) != 0 or rt.get("instance_count", 0) != 0:
        errors.append(
            f"ARM A control MUST NOT instantiate CowrieKVConnectorV1 "
            f"(store instances={st.get('instance_count')}, restore instances={rt.get('instance_count')})."
        )
    if store_res.get("verified_shards_count", 0) != 0 or restore_res.get("verified_shards_count", 0) != 0:
        errors.append("ARM A control MUST NOT write or read .state.cowrie shards.")

    s_toks = store_res.get("decode_token_ids", [])
    r_toks = restore_res.get("decode_token_ids", [])
    _check_greedy_parity("ARM A greedy token parity (store vs post-restart)", s_toks, r_toks, errors)
    g_toks = store_res.get("decode_gpu_hit_token_ids")
    if g_toks is not None:
        _check_greedy_parity("ARM A GPU-prefix-hit vs post-restart parity", g_toks, r_toks, errors)

    if errors:
        raise RuntimeError(
            "EXP-04 ARM A (CONTROL WITHOUT COWRIE) VALIDATION FAILED:\n  - " + "\n  - ".join(errors)
        )


def assert_connector_was_used(store_res: dict, restore_res: dict, hma_proof: dict) -> None:
    """Fail loudly if SupportsHMA is false, CowrieKVConnectorV1 was silently bypassed, or greedy parity broke."""
    errors: list[str] = []

    if not hma_proof.get("supports_hma_native_import"):
        errors.append("SupportsHMA was NOT imported natively from vllm (fallback stub active).")
    if not hma_proof.get("issubclass_supports_hma"):
        errors.append("issubclass(CowrieKVConnectorV1, SupportsHMA) is False.")
    if not hma_proof.get("has_request_finished_all_groups"):
        errors.append("CowrieKVConnectorV1 missing request_finished_all_groups.")

    st = store_res.get("connector_stats", {}).get("totals", {})
    rt = restore_res.get("connector_stats", {}).get("totals", {})

    # Phase 1 (Store) assertions
    if st.get("instance_count", 0) < 1:
        errors.append("Phase 1: zero CowrieKVConnectorV1 instances wrote stats (connector not initialized).")
    if not st.get("all_supports_hma", False):
        errors.append(f"Phase 1: live connector supports_hma check failed (totals={st}).")
    if st.get("save_kv_layer_calls", 0) <= 0:
        errors.append("Phase 1: save_kv_layer_calls == 0 (worker save hook never executed).")
    if st.get("saves_completed", 0) < 3:
        errors.append(
            f"Phase 1: expected >= 3 saved .state.cowrie shards, got saves_completed={st.get('saves_completed')}."
        )
    if st.get("saved_bytes", 0) <= 0 or st.get("saved_tokens", 0) <= 0:
        errors.append(
            f"Phase 1: zero saved KV payload (saved_tokens={st.get('saved_tokens')}, saved_bytes={st.get('saved_bytes')})."
        )
    if st.get("saves_failed", 0) > 0:
        errors.append(f"Phase 1: saves_failed={st.get('saves_failed')} (StatePage persistence errors).")
    if st.get("saves_skipped_inflight_cap", 0) > 0:
        errors.append(
            f"Phase 1: saves_skipped_inflight_cap={st.get('saves_skipped_inflight_cap')} "
            "(increase COWRIE_MAX_INFLIGHT_BYTES)."
        )
    if store_res.get("verified_shards_count", 0) < 3:
        errors.append(
            f"Phase 1: expected >= 3 hash-verified .state.cowrie shards on disk, "
            f"got {store_res.get('verified_shards_count')}."
        )

    # Phase 2 (Restore) assertions
    if rt.get("instance_count", 0) < 1:
        errors.append("Phase 2: zero CowrieKVConnectorV1 instances wrote stats (connector not initialized).")
    if not rt.get("all_supports_hma", False):
        errors.append(f"Phase 2: live connector supports_hma check failed (totals={rt}).")
    if rt.get("get_num_new_matched_tokens_calls", 0) <= 0:
        errors.append("Phase 2: get_num_new_matched_tokens_calls == 0 (scheduler hook never executed).")
    if rt.get("scheduler_hits", 0) < 3:
        errors.append(
            f"Phase 2: expected >= 3 Cowrie scheduler prefix hits across restart, got {rt.get('scheduler_hits')}."
        )
    if rt.get("start_load_kv_calls", 0) <= 0 or rt.get("loads_completed", 0) < 3:
        errors.append(
            f"Phase 2: Cowrie disk KV restore did not execute for all 3 prompts "
            f"(start_load_kv_calls={rt.get('start_load_kv_calls')}, loads_completed={rt.get('loads_completed')})."
        )
    if rt.get("loaded_tokens", 0) <= 0 or rt.get("loaded_bytes", 0) <= 0:
        errors.append(
            f"Phase 2: zero KV payload loaded (loaded_tokens={rt.get('loaded_tokens')}, loaded_bytes={rt.get('loaded_bytes')})."
        )
    if rt.get("loads_failed", 0) > 0:
        errors.append(f"Phase 2: loads_failed={rt.get('loads_failed')} (StatePage restore errors).")

    # Greedy token parity assertions (strict 100% match across restart — never weakened!)
    s_toks = store_res.get("decode_token_ids", [])
    r_toks = restore_res.get("decode_token_ids", [])
    _check_greedy_parity("ARM B greedy token parity (store vs Cowrie restore)", s_toks, r_toks, errors)
    g_toks = store_res.get("decode_gpu_hit_token_ids")
    if g_toks is not None:
        _check_greedy_parity("ARM B GPU-prefix-hit vs Cowrie restore parity", g_toks, r_toks, errors)

    if errors:
        raise RuntimeError(
            "EXP-04 FAIL-LOUD CHECK FAILED (connector bypassed, broken, or parity mismatch):\n  - "
            + "\n  - ".join(errors)
        )


def assert_cross_arm_parity(
    arm_a_store: dict, arm_a_restore: dict, arm_b_store: dict, arm_b_restore: dict
) -> None:
    """Assert 100% exact greedy token parity across ARM A (without Cowrie) and ARM B (with Cowrie)."""
    errors: list[str] = []
    _check_greedy_parity(
        "Cross-arm greedy parity (ARM A cold store vs ARM B Cowrie restore)",
        arm_a_store.get("decode_token_ids", []),
        arm_b_restore.get("decode_token_ids", []),
        errors,
    )
    _check_greedy_parity(
        "Cross-arm greedy parity (ARM A post-restart vs ARM B Cowrie restore)",
        arm_a_restore.get("decode_token_ids", []),
        arm_b_restore.get("decode_token_ids", []),
        errors,
    )
    if errors:
        raise RuntimeError(
            "EXP-04 CROSS-ARM GREEDY PARITY FAILED:\n  - " + "\n  - ".join(errors)
        )


def compute_arm_summary(
    arm_key: str,
    label: str,
    connector_enabled: bool,
    store_res: dict,
    restore_res: dict,
) -> dict:
    """Compute the full, identical metric set and connector counters for one experimental arm."""
    rt = restore_res["connector_stats"]["totals"]

    cold_ttft_ms = float(store_res["ttft_ms"])
    warm_ttft_ms = float(restore_res["ttft_ms"])
    ttft_speedup_x = round(cold_ttft_ms / max(warm_ttft_ms, 1e-3), 3)

    cold_decode_tokens = int(store_res["decode_output_tokens"])
    warm_decode_tokens = int(restore_res["decode_output_tokens"])
    cold_e2e_s = max(float(store_res["decode_e2e_ms"]) / 1000.0, 1e-6)
    warm_e2e_s = max(float(restore_res["decode_e2e_ms"]) / 1000.0, 1e-6)
    suffix_tokens = int(restore_res["new_suffix_output_tokens"])
    suffix_e2e_s = max(float(restore_res["new_suffix_e2e_ms"]) / 1000.0, 1e-6)

    warm_decode_only_s = max(warm_e2e_s - (warm_ttft_ms / 1000.0), 1e-6)
    warm_decode_only_tok_s = round(max(warm_decode_tokens - 1, 1) / warm_decode_only_s, 2)

    restore_prompt_tokens_total = (
        int(restore_res["ttft_prompt_tokens"])
        + int(restore_res["decode_prompt_tokens"])
        + int(restore_res["new_suffix_prompt_tokens"])
    )
    loaded_tokens = int(rt.get("loaded_tokens", 0))
    matched_tokens = int(rt.get("scheduler_matched_tokens", 0))
    kv_hit_rate_loaded_pct = round(100.0 * loaded_tokens / max(restore_prompt_tokens_total, 1), 2)
    kv_hit_rate_scheduler_pct = round(100.0 * matched_tokens / max(restore_prompt_tokens_total, 1), 2)
    request_hit_rate_pct = round(100.0 * min(int(rt.get("scheduler_hits", 0)), 3) / 3.0, 2)

    vllm_reported_cached = restore_res.get("vllm_num_cached_tokens_total")
    vllm_reported_hit_pct = (
        round(100.0 * int(vllm_reported_cached) / max(restore_prompt_tokens_total, 1), 2)
        if vllm_reported_cached is not None
        else None
    )

    greedy_parity = store_res["decode_token_ids"] == restore_res["decode_token_ids"]
    gpu_hit_parity = (
        store_res["decode_gpu_hit_token_ids"] == restore_res["decode_token_ids"]
        if store_res.get("decode_gpu_hit_token_ids") is not None
        else None
    )

    if connector_enabled:
        verdict = "PASS" if (greedy_parity and warm_ttft_ms < cold_ttft_ms) else "FAIL_OR_NO_SPEEDUP"
    else:
        verdict = (
            "CONTROL_NO_CROSS_RESTART_HIT"
            if (greedy_parity and int(vllm_reported_cached or 0) == 0)
            else ("CONTROL_UNEXPECTED_RESTART_HIT" if greedy_parity else "CONTROL_PARITY_FAIL")
        )

    return {
        "arm": arm_key,
        "label": label,
        "connector_enabled": connector_enabled,
        "native_prefix_caching": True,
        "engine_init_s": {
            "phase_1_store": store_res["engine_init_s"],
            "phase_2_restore": restore_res["engine_init_s"],
        },
        "subprocess_wall_s": {
            "phase_1_store": store_res.get("subprocess_wall_s"),
            "phase_2_restore": restore_res.get("subprocess_wall_s"),
        },
        "metrics": {
            "prompt_tokens_ttft": int(restore_res["ttft_prompt_tokens"]),
            "prompt_tokens_decode": int(restore_res["decode_prompt_tokens"]),
            "prompt_tokens_new_suffix": int(restore_res["new_suffix_prompt_tokens"]),
            "restore_prompt_tokens_total": restore_prompt_tokens_total,
            "cold_ttft_ms": round(cold_ttft_ms, 2),
            "warm_ttft_ms": round(warm_ttft_ms, 2),
            "ttft_speedup_x": ttft_speedup_x,
            "cold_decode_e2e_ms": round(float(store_res["decode_e2e_ms"]), 2),
            "warm_decode_e2e_ms": round(float(restore_res["decode_e2e_ms"]), 2),
            "cold_e2e_tokens_per_s": round(cold_decode_tokens / cold_e2e_s, 2),
            "warm_e2e_tokens_per_s": round(warm_decode_tokens / warm_e2e_s, 2),
            "warm_decode_only_tokens_per_s": warm_decode_only_tok_s,
            "new_suffix_warm_e2e_ms": round(float(restore_res["new_suffix_e2e_ms"]), 2),
            "new_suffix_warm_e2e_tokens_per_s": round(suffix_tokens / suffix_e2e_s, 2),
            "kv_cache_hit_rate_loaded_pct": kv_hit_rate_loaded_pct,
            "kv_cache_hit_rate_scheduler_pct": kv_hit_rate_scheduler_pct,
            "kv_cache_request_hit_rate_pct": request_hit_rate_pct,
            "vllm_num_cached_tokens_total": vllm_reported_cached,
            "vllm_reported_kv_hit_rate_pct": vllm_reported_hit_pct,
            "greedy_token_parity": greedy_parity,
            "gpu_prefix_hit_parity": gpu_hit_parity,
            "decode_output_tokens": warm_decode_tokens,
            "state_shards_count": int(store_res["verified_shards_count"]),
            "state_shards_total_bytes": int(store_res["state_bytes_total"]),
            "hmac_sha256_verified": bool(store_res["all_shards_hmac_verified"]),
        },
        "connector_counters": {
            "phase_1_store": store_res["connector_stats"],
            "phase_2_restore": restore_res["connector_stats"],
        },
        "verdict": verdict,
    }


def compute_ab_benchmark_summary(
    arm_a_store: dict,
    arm_a_restore: dict,
    arm_b_store: dict,
    arm_b_restore: dict,
    hma_proof: dict,
    gpu_info: dict,
) -> dict:
    """Build ONE unified paired A/B receipt with explicit arm_a, arm_b, and ab delta_table."""
    arm_a = compute_arm_summary(
        arm_key="arm_a_without_cowrie",
        label="Control (without Cowrie): plain vLLM + native prefix caching ON",
        connector_enabled=False,
        store_res=arm_a_store,
        restore_res=arm_a_restore,
    )
    arm_b = compute_arm_summary(
        arm_key="arm_b_with_cowrie",
        label="Treatment (with Cowrie): vLLM + CowrieKVConnectorV1 + native prefix caching ON",
        connector_enabled=True,
        store_res=arm_b_store,
        restore_res=arm_b_restore,
    )
    ma = arm_a["metrics"]
    mb = arm_b["metrics"]

    a_toks = arm_a_restore["decode_token_ids"]
    b_toks = arm_b_restore["decode_token_ids"]
    cross_arm_exact = (
        arm_a_store["decode_token_ids"] == b_toks
        and a_toks == b_toks
    )
    first_diff_idx = next((i for i, (x, y) in enumerate(zip(a_toks, b_toks)) if x != y), None)
    if first_diff_idx is None and len(a_toks) != len(b_toks):
        first_diff_idx = min(len(a_toks), len(b_toks))
    cross_arm_matching_prefix = len(a_toks) if first_diff_idx is None else first_diff_idx
    cross_arm_first_diff = (
        None
        if first_diff_idx is None
        else {
            "token_index": first_diff_idx,
            "arm_a_token_ids_window": a_toks[first_diff_idx:first_diff_idx + 4],
            "arm_b_token_ids_window": b_toks[first_diff_idx:first_diff_idx + 4],
        }
    )
    both_arms_parity = bool(ma["greedy_token_parity"] and mb["greedy_token_parity"])

    arm_a_cached = int(ma["vllm_num_cached_tokens_total"] or 0)
    arm_a_survived = arm_a_cached > 0
    if not arm_a_survived:
        arm_a_restart_summary = (
            "ARM A (plain vLLM with native prefix caching ON and no Cowrie connector) cannot produce a warm KV "
            f"hit across an OS process restart because vLLM's in-GPU prefix cache is wiped on process exit "
            f"(vllm_num_cached_tokens_total=0, cold TTFT={ma['cold_ttft_ms']:.2f} ms, "
            f"post-restart TTFT={ma['warm_ttft_ms']:.2f} ms, speedup={ma['ttft_speedup_x']:.3f}x)."
        )
    else:
        arm_a_restart_summary = (
            f"ARM A reported {arm_a_cached} cached tokens across process restart "
            f"(cold TTFT={ma['cold_ttft_ms']:.2f} ms, post-restart TTFT={ma['warm_ttft_ms']:.2f} ms)."
        )

    warm_ttft_speedup_cowrie_vs_native = round(ma["warm_ttft_ms"] / max(mb["warm_ttft_ms"], 1e-3), 3)

    def _row(metric: str, unit: str, lower_is_better: bool | None) -> dict:
        va = ma[metric]
        vb = mb[metric]
        delta = round(vb - va, 3) if isinstance(va, (int, float)) and isinstance(vb, (int, float)) else None
        speedup = None
        if isinstance(va, (int, float)) and isinstance(vb, (int, float)) and va > 0 and vb > 0:
            if lower_is_better is True:
                speedup = round(va / vb, 3)
            elif lower_is_better is False:
                speedup = round(vb / va, 3)
        return {
            "metric": metric,
            "unit": unit,
            "arm_a_without_cowrie": va,
            "arm_b_with_cowrie": vb,
            "delta_b_minus_a": delta,
            "speedup_b_vs_a_x": speedup,
        }

    delta_table = [
        _row("cold_ttft_ms", "ms", True),
        _row("warm_ttft_ms", "ms", True),
        _row("ttft_speedup_x", "x", False),
        _row("cold_decode_e2e_ms", "ms", True),
        _row("warm_decode_e2e_ms", "ms", True),
        _row("cold_e2e_tokens_per_s", "tok/s", False),
        _row("warm_e2e_tokens_per_s", "tok/s", False),
        _row("warm_decode_only_tokens_per_s", "tok/s", False),
        _row("new_suffix_warm_e2e_ms", "ms", True),
        _row("new_suffix_warm_e2e_tokens_per_s", "tok/s", False),
        _row("kv_cache_hit_rate_loaded_pct", "%", False),
        _row("kv_cache_hit_rate_scheduler_pct", "%", False),
        _row("kv_cache_request_hit_rate_pct", "%", False),
        _row("vllm_num_cached_tokens_total", "tokens", False),
        _row("vllm_reported_kv_hit_rate_pct", "%", False),
        _row("decode_output_tokens", "tokens", None),
        _row("state_shards_count", "shards", None),
        _row("state_shards_total_bytes", "bytes", None),
    ]

    overall_pass = (
        both_arms_parity
        and arm_b["verdict"] == "PASS"
        and mb["warm_ttft_ms"] < ma["warm_ttft_ms"]
    )

    return {
        "experiment": "EXP-04",
        "lane": "CLOUD-LANE-01",
        "design": "paired_ab_cross_restart",
        "hardware": gpu_info,
        "config": CONFIG,
        "hma_proof": hma_proof,
        "arm_a_without_cowrie": arm_a,
        "arm_b_with_cowrie": arm_b,
        "ab": {
            "comparison": (
                "ARM B (with CowrieKVConnectorV1 + native prefix caching ON) vs "
                "ARM A (control without Cowrie, plain vLLM + native prefix caching ON) "
                "on identical GPU, model, prompts, max_model_len, gpu_memory_utilization, and seed"
            ),
            "arm_a_survived_process_restart": arm_a_survived,
            "arm_a_restart_summary": arm_a_restart_summary,
            "arm_a_greedy_token_parity": bool(ma["greedy_token_parity"]),
            "arm_b_greedy_token_parity": bool(mb["greedy_token_parity"]),
            "both_arms_greedy_token_parity": both_arms_parity,
            "cross_arm_exact_token_match": cross_arm_exact,
            "cross_arm_matching_prefix_tokens": cross_arm_matching_prefix,
            "cross_arm_first_diff": cross_arm_first_diff,
            "warm_ttft_speedup_cowrie_vs_native_x": warm_ttft_speedup_cowrie_vs_native,
            "warm_ttft_delta_ms": round(mb["warm_ttft_ms"] - ma["warm_ttft_ms"], 2),
            "warm_decode_e2e_delta_ms": round(mb["warm_decode_e2e_ms"] - ma["warm_decode_e2e_ms"], 2),
            "warm_e2e_tokens_per_s_delta": round(mb["warm_e2e_tokens_per_s"] - ma["warm_e2e_tokens_per_s"], 2),
            "warm_decode_only_tokens_per_s_delta": round(
                mb["warm_decode_only_tokens_per_s"] - ma["warm_decode_only_tokens_per_s"], 2
            ),
            "new_suffix_warm_e2e_delta_ms": round(
                mb["new_suffix_warm_e2e_ms"] - ma["new_suffix_warm_e2e_ms"], 2
            ),
            "new_suffix_warm_e2e_tokens_per_s_delta": round(
                mb["new_suffix_warm_e2e_tokens_per_s"] - ma["new_suffix_warm_e2e_tokens_per_s"], 2
            ),
            "kv_hit_rate_loaded_delta_pct": round(
                mb["kv_cache_hit_rate_loaded_pct"] - ma["kv_cache_hit_rate_loaded_pct"], 2
            ),
            "vllm_cached_tokens_delta": int(mb["vllm_num_cached_tokens_total"] or 0) - arm_a_cached,
            "delta_table": delta_table,
        },
        "verdict": "PASS" if overall_pass else "FAIL_OR_NO_SPEEDUP",
    }


PROMPTS = build_benchmark_prompts(PROMPT_ROWS)
print(
    "Prepared 4 identical benchmark prompt strings:",
    {k: len(v) for k, v in PROMPTS.items()},
)

## 1. GPU Preflight, Memory-Fit Gate & Dependency Installation (Colab Ultra)

- Prints the active profile's exact parameter count and computed weight footprint, queries `nvidia-smi` for total/free VRAM and compute capability, and **rejects any profile that cannot fit the selected GPU before downloading or loading weights**.
- Installs `vllm>=0.15.0` (upgrading to latest so Qwen3.8/Qwen3.6/Inkling-2/GLM-4.7-Flash architectures are supported) and builds/installs `cowrie-ai` directly from this repository (installing a minimal Rust toolchain via `rustup` if `cargo` is absent).
- Uses zero paid GCP resources and zero external cloud credentials.

In [ ]:
import json
import shutil
import subprocess

print(
    f"Selected profile {MODEL_PROFILE!r}: {PROFILE['model_id']} | "
    f"params={PROFILE['params']:,} (active={PROFILE['active_params']:,}) | "
    f"weight_dtype={PROFILE['weight_dtype']} | "
    f"computed_weight={PROFILE['computed_weight_gib']:.2f} GiB ({PROFILE['computed_weight_gb']:.2f} GB) | "
    f"min_required_vram={PROFILE['min_required_vram_gib']:.2f} GiB"
)

if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No NVIDIA GPU detected (nvidia-smi not found). "
        "Attach an H100 80GB or RTX PRO 6000 Blackwell 96GB runtime in Colab Ultra."
    )

smi_out = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version,compute_cap,power.limit",
        "--format=csv,noheader,nounits",
    ],
    text=True,
).strip()
first_gpu = [x.strip() for x in smi_out.splitlines()[0].split(",")]
gpu_name, total_mib_s, free_mib_s, driver_ver, compute_cap, power_limit_w = first_gpu
total_vram_gib = round(float(total_mib_s) / 1024.0, 2)
free_vram_gib = round(float(free_mib_s) / 1024.0, 2)

if "H100" in gpu_name.upper():
    detected_profile = "H100_80GB_HOPPER"
elif "PRO 6000" in gpu_name.upper() or compute_cap.startswith("12."):
    detected_profile = "RTX_PRO_6000_BLACKWELL"
else:
    detected_profile = f"OTHER_CUDA_SM_{compute_cap.replace('.', '')}"

# Fit gate: reject any profile that cannot fit the selected GPU before downloading/loading weights
FIT_REPORT = check_model_fits_gpu(PROFILE, total_vram_gib, free_vram_gib, compute_cap)
assert FIT_REPORT["fits"] is True

GPU_INFO = {
    "gpu_name": gpu_name,
    "vram_total_gib": total_vram_gib,
    "vram_free_gib": free_vram_gib,
    "driver_version": driver_ver,
    "compute_capability": compute_cap,
    "power_limit_w": power_limit_w,
    "detected_profile": detected_profile,
    "memory_fit": FIT_REPORT,
}
print("GPU Preflight & Memory Fit: PASS")
print(json.dumps(GPU_INFO, indent=2))

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import shutil
import subprocess
import sys


def _run(cmd, env=None, cwd=None):
    print("+", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=cwd)


def _version_tuple(v: str) -> tuple[int, ...]:
    parts = []
    for tok in v.split("."):
        digits = "".join(ch for ch in tok if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)


# 1. Ensure vLLM 0.15+ (latest release) is installed
need_vllm = True
try:
    cur_vllm = importlib.metadata.version("vllm")
    if _version_tuple(cur_vllm) >= (0, 15, 0):
        print(f"vLLM already installed: {cur_vllm}")
        need_vllm = False
    else:
        print(f"Upgrading vLLM from {cur_vllm} to >=0.15.0...")
except importlib.metadata.PackageNotFoundError:
    print("Installing vllm>=0.15.0...")

if need_vllm:
    _run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "vllm>=0.15.0"])

# 2. Ensure minimal Rust cargo toolchain is available to compile libcowrie_core.so
cargo_bin_dir = Path.home() / ".cargo" / "bin"
if cargo_bin_dir.is_dir() and str(cargo_bin_dir) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{cargo_bin_dir}:{os.environ.get('PATH', '')}"

if not shutil.which("cargo"):
    print("Rust cargo not found; installing minimal rustup toolchain...")
    _run([
        "bash",
        "-lc",
        "curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal --default-toolchain stable",
    ])
    os.environ["PATH"] = f"{cargo_bin_dir}:{os.environ.get('PATH', '')}"

assert shutil.which("cargo"), "cargo still missing after rustup installation!"

# 3. Locate local Cowrie checkout or clone from COWRIE_REPO_URL
cowrie_repo_dir = None
for cand in (Path.cwd(), Path.cwd().parent, Path("/content/cowrie")):
    if (cand / "Cargo.toml").is_file() and (cand / "adapters" / "vllm" / "cowrie_kv_connector.py").is_file():
        cowrie_repo_dir = cand.resolve()
        break

if cowrie_repo_dir is None:
    cowrie_repo_dir = Path("/content/cowrie") if Path("/content").is_dir() else Path("/tmp/cowrie-src")
    if cowrie_repo_dir.exists():
        shutil.rmtree(cowrie_repo_dir)
    try:
        _run(["git", "clone", "--depth", "1", "--branch", COWRIE_GIT_REF, COWRIE_REPO_URL, str(cowrie_repo_dir)])
    except subprocess.CalledProcessError:
        print(f"Branch {COWRIE_GIT_REF!r} not found on remote; falling back to main...")
        if cowrie_repo_dir.exists():
            shutil.rmtree(cowrie_repo_dir)
        _run(["git", "clone", "--depth", "1", "--branch", "main", COWRIE_REPO_URL, str(cowrie_repo_dir)])

COWRIE_GIT_SHA = subprocess.check_output(
    ["git", "-C", str(cowrie_repo_dir), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Using Cowrie repo at {cowrie_repo_dir} (HEAD {COWRIE_GIT_SHA})")

# 4. Install cowrie-ai Python package without touching installed torch/vllm deps
_run([sys.executable, "-m", "pip", "install", "-q", "setuptools>=68", "wheel"])
build_env = os.environ.copy()
build_env["CARGO_TARGET_DIR"] = "/tmp/cowrie-cargo-target"
_run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "--no-deps",
        "--force-reinstall",
        str(cowrie_repo_dir),
    ],
    env=build_env,
)

# 5. Verify native Cowrie library + vllm.general_plugins entry point
import cowrie

eps = [ep.name for ep in importlib.metadata.entry_points(group="vllm.general_plugins")]
print("Installed cowrie-ai:", importlib.metadata.version("cowrie-ai"))
print("Cowrie SHA-NI hardware accel:", bool(cowrie.has_shani()))
print("vllm.general_plugins entry points:", eps)
assert "cowrie_kv_connector" in eps, (
    f"cowrie_kv_connector entry point missing from vllm.general_plugins: {eps}"
)

## 2. Register `CowrieKVConnectorV1` & Prove `SupportsHMA` at Runtime

This cell resolves `CowrieKVConnectorV1` **through vLLM's own `KVConnectorFactory`** (testing `vllm.general_plugins` auto-discovery first) and asserts that it genuinely subclasses vLLM's real `SupportsHMA` mixin — never the fallback stub.

In [ ]:
import importlib.metadata
import json
import vllm
from vllm.config import KVTransferConfig
from vllm.distributed.kv_transfer.kv_connector.factory import KVConnectorFactory
from vllm.distributed.kv_transfer.kv_connector.v1.base import (
    KVConnectorBase_V1,
    SupportsHMA,
)

vllm_ver = getattr(vllm, "__version__", importlib.metadata.version("vllm"))
ver_parts = tuple(int("".join(c for c in p if c.isdigit()) or "0") for p in vllm_ver.split(".")[:3])
assert ver_parts >= (0, 15, 0), f"Expected vLLM >= 0.15.0, got {vllm_ver}"

# Trigger vLLM's real general plugin discovery
try:
    from vllm.plugins import load_general_plugins
    load_general_plugins()
except Exception as exc:
    print(f"Note: load_general_plugins raised {exc!r}; checking explicit registration...")

registry = getattr(KVConnectorFactory, "_registry", {})
auto_registered = "CowrieKVConnectorV1" in registry
if not auto_registered:
    from cowrie_vllm_adapters._plugin import register_vllm_plugin
    register_vllm_plugin()

assert "CowrieKVConnectorV1" in getattr(KVConnectorFactory, "_registry", {}), (
    "CowrieKVConnectorV1 failed to register in KVConnectorFactory._registry!"
)

# Resolve through vLLM's factory (exact path used by EngineCore)
kv_probe_cfg = KVTransferConfig(kv_connector="CowrieKVConnectorV1", kv_role="kv_both")
resolved_cls = KVConnectorFactory.get_connector_class(kv_probe_cfg)

from cowrie_vllm_adapters.cowrie_kv_connector import (
    CowrieKVConnectorV1,
    SUPPORTS_HMA_NATIVE,
    read_connector_stats,
    reset_connector_stats,
)

assert resolved_cls is CowrieKVConnectorV1, f"Unexpected factory class: {resolved_cls}"
assert issubclass(resolved_cls, KVConnectorBase_V1), "Not a KVConnectorBase_V1 subclass!"
assert SUPPORTS_HMA_NATIVE is True, (
    "SUPPORTS_HMA_NATIVE is False — CowrieKVConnectorV1 fell back to stub _SupportsHMA!"
)
assert issubclass(resolved_cls, SupportsHMA), (
    f"Runtime SupportsHMA check FAILED! MRO={[c.__name__ for c in resolved_cls.__mro__]}"
)
has_all_groups = callable(getattr(resolved_cls, "request_finished_all_groups", None))
assert has_all_groups, "request_finished_all_groups method missing on CowrieKVConnectorV1!"

HMA_PROOF = {
    "vllm_version": vllm_ver,
    "cowrie_git_sha": COWRIE_GIT_SHA,
    "auto_registered_via_entry_point": auto_registered,
    "factory_resolved_class": f"{resolved_cls.__module__}.{resolved_cls.__qualname__}",
    "mro": [f"{c.__module__}.{c.__qualname__}" for c in resolved_cls.__mro__],
    "supports_hma_class": f"{SupportsHMA.__module__}.{SupportsHMA.__qualname__}",
    "supports_hma_native_import": SUPPORTS_HMA_NATIVE,
    "issubclass_supports_hma": issubclass(resolved_cls, SupportsHMA),
    "has_request_finished_all_groups": has_all_groups,
}
print("SupportsHMA RUNTIME PROOF: PASS")
print(json.dumps(HMA_PROOF, indent=2))

## 3. Paired A/B Cross-Restart Benchmark (Identical GPU, Model, Prompts, Limits & Seed)

We write a standalone worker script `/tmp/cowrie_exp04_worker.py` and run **both paired arms** on the exact same GPU, model, prompts, `max_model_len=8192`, `gpu_memory_utilization=0.90`, and `seed=42`, each across **two completely separate OS processes**:

1. **ARM A (`control_without_cowrie`, plain vLLM + native prefix caching ON, NO Cowrie connector):**
   - **Process 1 (`--phase store`):** Cold GPU VRAM. Measures cold TTFT, cold decode (64 greedy tokens + parity reference), and seeds the 3rd prefix. Exits completely (100% VRAM teardown).
   - **Process 2 (`--phase restore`):** Fresh OS process relying solely on vLLM's native prefix caching. Honestly records post-restart TTFT, decode tok/s, new-suffix E2E, `vllm_num_cached_tokens_total` (0 when volatile VRAM cache does not survive process exit), greedy parity, and zeroed connector counters.
2. **ARM B (`with_cowrie`, vLLM + `CowrieKVConnectorV1` + native prefix caching ON):**
   - **Process 1 (`--phase store`):** Cold GPU VRAM, empty `STATE_DIR`. Persists 3 HMAC-SHA256 `.state.cowrie` shards on disk and verifies every shard. Exits completely (100% VRAM teardown).
   - **Process 2 (`--phase restore`):** Fresh OS process, 0% in-GPU prefix cache, same `STATE_DIR`. Restores all 3 prefixes from `.state.cowrie` shards on disk (including a brand-new suffix question on the 3rd corpus) and enforces strict `64/64` greedy parity both within ARM B and against ARM A.

In [ ]:
from pathlib import Path

WORKER_SCRIPT_PATH = Path("/tmp/cowrie_exp04_worker.py")

WORKER_SOURCE = r'''#!/usr/bin/env python3
"""Standalone single-phase worker for EXP-04 paired A/B (separate OS processes for store vs restore)."""
from __future__ import annotations

import argparse
import gc
import json
import os
from pathlib import Path
import time


def _extract_metrics_optional(out) -> dict:
    m = getattr(out, "metrics", None)
    info = {
        "num_cached_tokens": getattr(out, "num_cached_tokens", None),
        "vllm_ttft_ms": None,
    }
    if m is not None:
        arr = getattr(m, "arrival_time", None)
        ft = getattr(m, "first_token_time", None)
        if arr is not None and ft is not None and ft >= arr:
            info["vllm_ttft_ms"] = round((ft - arr) * 1000.0, 2)
    return info


def _shutdown_llm(llm) -> None:
    try:
        eng = getattr(llm, "llm_engine", None)
        ec = getattr(eng, "engine_core", None)
        if ec is not None and hasattr(ec, "shutdown"):
            ec.shutdown()
    except Exception:
        pass
    del llm
    gc.collect()


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--arm", choices=["control_without_cowrie", "with_cowrie"], required=True)
    parser.add_argument("--phase", choices=["store", "restore"], required=True)
    parser.add_argument("--state-dir", required=True)
    parser.add_argument("--config-json", required=True)
    parser.add_argument("--prompts-json", required=True)
    parser.add_argument("--result-json", required=True)
    args = parser.parse_args()

    os.environ["VLLM_USE_V1"] = "1"
    os.environ["VLLM_NO_USAGE_STATS"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    use_cowrie = args.arm == "with_cowrie"
    cfg = json.loads(Path(args.config_json).read_text(encoding="utf-8"))
    prompts = json.loads(Path(args.prompts_json).read_text(encoding="utf-8"))
    seed = int(cfg.get("seed", 42))
    state_dir = Path(args.state_dir)
    state_dir.mkdir(mode=0o700, parents=True, exist_ok=True)

    from vllm import LLM, SamplingParams

    # For ARM B (with_cowrie), reset per-phase stats files before engine start.
    # For ARM A (control_without_cowrie), do not import Cowrie at all while vLLM runs.
    if use_cowrie:
        from cowrie_vllm_adapters.cowrie_kv_connector import reset_connector_stats
        reset_connector_stats(state_dir)

    llm_kwargs = dict(
        model=cfg["model_id"],
        dtype=cfg["dtype"],
        max_model_len=int(cfg["max_model_len"]),
        gpu_memory_utilization=float(cfg["gpu_memory_utilization"]),
        trust_remote_code=bool(cfg.get("trust_remote_code", True)),
        enable_prefix_caching=True,
        enforce_eager=bool(cfg["enforce_eager"]),
        seed=seed,
    )
    if cfg.get("quantization"):
        llm_kwargs["quantization"] = cfg["quantization"]

    if use_cowrie:
        import cowrie
        from cowrie_vllm_adapters._plugin import register_vllm_plugin
        from vllm.config import KVTransferConfig

        register_vllm_plugin()
        llm_kwargs["kv_transfer_config"] = KVTransferConfig(
            kv_connector="CowrieKVConnectorV1",
            kv_role="kv_both",
            kv_connector_extra_config={
                "cowrie_state_dir": str(state_dir),
                "cowrie_max_inflight_bytes": int(cfg["cowrie_max_inflight_bytes"]),
                "cowrie_max_cache_bytes": int(cfg["cowrie_max_cache_bytes"]),
            },
        )

    t_init0 = time.perf_counter()
    llm = LLM(**llm_kwargs)
    engine_init_s = round(time.perf_counter() - t_init0, 2)

    # Apply tokenizer chat_template when present so instruct/chat checkpoints produce coherent outputs
    # while preserving exact shared prefix alignment for the new-suffix corpus.
    tok = llm.get_tokenizer()
    block_size = 16
    try:
        bs = getattr(llm.llm_engine.vllm_config.cache_config, "block_size", None)
        if bs:
            block_size = int(bs)
    except Exception:
        pass

    def _fmt_chat(text: str) -> str:
        if getattr(tok, "chat_template", None):
            return tok.apply_chat_template(
                [{"role": "user", "content": text}],
                tokenize=False,
                add_generation_prompt=True,
            )
        return text

    p_ttft = _fmt_chat(prompts["prompt_ttft"])
    p_decode = _fmt_chat(prompts["prompt_decode"])
    p_new_suffix = _fmt_chat(prompts["prompt_new_suffix"])
    seed_raw = prompts["prompt_prefix_seed"]
    idx_seed = p_new_suffix.find(seed_raw)
    if idx_seed >= 0:
        p_prefix_seed = p_new_suffix[: idx_seed + len(seed_raw)]
    else:
        p_prefix_seed = seed_raw
        p_new_suffix = prompts["prompt_new_suffix"]

    seed_ids = tok.encode(p_prefix_seed)
    new_suffix_ids = tok.encode(p_new_suffix)
    aligned_n = ((len(seed_ids) - 1) // block_size) * block_size
    assert aligned_n > 0 and new_suffix_ids[:aligned_n] == seed_ids[:aligned_n], (
        f"BPE boundary mismatch on prefix_suffix corpus (aligned_n={aligned_n}, block_size={block_size})!"
    )

    sp_warmup = SamplingParams(temperature=0.0, seed=seed, max_tokens=4)
    sp_ttft = SamplingParams(temperature=0.0, seed=seed, max_tokens=1)
    decode_n = int(cfg["decode_max_tokens"])
    sp_decode = SamplingParams(temperature=0.0, seed=seed, ignore_eos=True, max_tokens=decode_n)

    # Short warmup (< block_size tokens) so CUDA kernels are warm without polluting prefix cache
    llm.generate(["Warmup."], sp_warmup, use_tqdm=False)

    # 1. TTFT probe (disjoint prefix A, max_tokens=1)
    t0 = time.perf_counter()
    out_ttft = llm.generate([p_ttft], sp_ttft, use_tqdm=False)[0]
    ttft_ms = (time.perf_counter() - t0) * 1000.0
    m_ttft = _extract_metrics_optional(out_ttft)

    # 2. Decode & greedy parity probe (disjoint prefix B, exact decode_n tokens)
    t1 = time.perf_counter()
    out_dec = llm.generate([p_decode], sp_decode, use_tqdm=False)[0]
    decode_e2e_ms = (time.perf_counter() - t1) * 1000.0
    m_dec = _extract_metrics_optional(out_dec)

    # In store phase, also record an in-GPU prefix-cache hit reference for exact cross-check
    decode_gpu_hit_ids = None
    if args.phase == "store":
        out_dec_gpu = llm.generate([p_decode], sp_decode, use_tqdm=False)[0]
        decode_gpu_hit_ids = list(out_dec_gpu.outputs[0].token_ids)

    # 3. Prefix seed (store) OR brand-new suffix reuse (restore) on disjoint prefix C
    if args.phase == "store":
        t2 = time.perf_counter()
        out_p3 = llm.generate([p_prefix_seed], sp_ttft, use_tqdm=False)[0]
        p3_ms = (time.perf_counter() - t2) * 1000.0
    else:
        t2 = time.perf_counter()
        out_p3 = llm.generate([p_new_suffix], sp_decode, use_tqdm=False)[0]
        p3_ms = (time.perf_counter() - t2) * 1000.0
    m_p3 = _extract_metrics_optional(out_p3)

    # Cleanly shut down EngineCore (triggers connector.shutdown() -> io_pool.shutdown(wait=True) if enabled)
    _shutdown_llm(llm)

    from cowrie_vllm_adapters.cowrie_kv_connector import read_connector_stats

    if use_cowrie and args.phase == "store":
        deadline = time.perf_counter() + 30.0
        while time.perf_counter() < deadline:
            cur_stats = read_connector_stats(state_dir)["totals"]
            if (
                cur_stats.get("inflight_saves", 0) == 0
                and (
                    cur_stats.get("saves_completed", 0) >= 3
                    or cur_stats.get("saves_failed", 0) > 0
                    or cur_stats.get("saves_skipped_inflight_cap", 0) > 0
                )
            ):
                break
            time.sleep(0.2)

    connector_stats = read_connector_stats(state_dir)
    shards = sorted(state_dir.glob("*.state.cowrie"))
    verified_count = 0
    all_hmac = False
    if use_cowrie and shards:
        import cowrie
        all_hmac = True
        for sp in shards:
            info = cowrie.verify_statepage_file(
                str(sp),
                expected_runtime_id="vllm-v1",
                verify_hash=True,
            )
            verified_count += 1
            all_hmac = all_hmac and bool(info.get("hmac_verified", False))

    cached_vals = [
        x for x in (m_ttft["num_cached_tokens"], m_dec["num_cached_tokens"], m_p3["num_cached_tokens"])
        if x is not None
    ]
    vllm_cached_total = sum(cached_vals) if len(cached_vals) == 3 else None

    result = {
        "arm": args.arm,
        "phase": args.phase,
        "seed": seed,
        "engine_init_s": engine_init_s,
        "ttft_ms": round(ttft_ms, 2),
        "ttft_prompt_tokens": len(out_ttft.prompt_token_ids),
        "ttft_vllm_metrics": m_ttft,
        "decode_e2e_ms": round(decode_e2e_ms, 2),
        "decode_prompt_tokens": len(out_dec.prompt_token_ids),
        "decode_output_tokens": len(out_dec.outputs[0].token_ids),
        "decode_token_ids": list(out_dec.outputs[0].token_ids),
        "decode_gpu_hit_token_ids": decode_gpu_hit_ids,
        "decode_text_preview": out_dec.outputs[0].text[:200],
        "decode_vllm_metrics": m_dec,
        "new_suffix_e2e_ms": round(p3_ms, 2),
        "new_suffix_prompt_tokens": len(out_p3.prompt_token_ids),
        "new_suffix_output_tokens": len(out_p3.outputs[0].token_ids),
        "new_suffix_vllm_metrics": m_p3,
        "vllm_num_cached_tokens_total": vllm_cached_total,
        "state_shards": [s.name for s in shards],
        "verified_shards_count": verified_count,
        "state_bytes_total": sum(s.stat().st_size for s in shards),
        "all_shards_hmac_verified": all_hmac,
        "connector_stats": connector_stats,
    }
    Path(args.result_json).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"ARM {args.arm.upper()} PHASE {args.phase.upper()} COMPLETE -> {args.result_json}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

WORKER_SCRIPT_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
print(f"Wrote {WORKER_SCRIPT_PATH} ({len(WORKER_SOURCE)} bytes)")

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

# Separate clean directories for ARM A (unused control dir) and ARM B (Cowrie StatePage dir)
ARM_A_STATE_DIR = Path(f"{STATE_DIR}_arm_a_control_unused")
ARM_B_STATE_DIR = STATE_DIR
for d in (ARM_A_STATE_DIR, ARM_B_STATE_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(mode=0o700, parents=True, exist_ok=True)

cfg_file = Path("/tmp/cowrie_exp04_cfg.json")
prompts_file = Path("/tmp/cowrie_exp04_prompts.json")
cfg_file.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
prompts_file.write_text(json.dumps(PROMPTS, indent=2), encoding="utf-8")

worker_env = os.environ.copy()
worker_env["COWRIE_HMAC_KEY"] = COWRIE_HMAC_KEY_HEX


def _run_phase(arm: str, phase: str, state_dir: Path, out_file: Path) -> dict:
    if out_file.exists():
        out_file.unlink()
    cmd = [
        sys.executable,
        str(WORKER_SCRIPT_PATH),
        "--arm", arm,
        "--phase", phase,
        "--state-dir", str(state_dir),
        "--config-json", str(cfg_file),
        "--prompts-json", str(prompts_file),
        "--result-json", str(out_file),
    ]
    print(f"\n========== LAUNCHING OS SUBPROCESS: {arm.upper()} / PHASE {phase.upper()} ==========", flush=True)
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, env=worker_env)
    wall_s = round(time.perf_counter() - t0, 2)
    if proc.returncode != 0:
        raise RuntimeError(f"{arm} phase {phase} subprocess exited with rc={proc.returncode}")
    assert out_file.is_file(), f"{arm} phase {phase} did not produce {out_file}"
    data = json.loads(out_file.read_text(encoding="utf-8"))
    data["subprocess_wall_s"] = wall_s
    return data


# ARM A: Control (without Cowrie) — plain vLLM + native prefix caching ON
ARM_A_STORE_RES = _run_phase(
    "control_without_cowrie", "store", ARM_A_STATE_DIR, Path("/tmp/cowrie_exp04_arm_a_store.json")
)
ARM_A_RESTORE_RES = _run_phase(
    "control_without_cowrie", "restore", ARM_A_STATE_DIR, Path("/tmp/cowrie_exp04_arm_a_restore.json")
)
assert_control_arm_valid(ARM_A_STORE_RES, ARM_A_RESTORE_RES)

# ARM B: Treatment (with Cowrie) — vLLM + CowrieKVConnectorV1 + native prefix caching ON
ARM_B_STORE_RES = _run_phase(
    "with_cowrie", "store", ARM_B_STATE_DIR, Path("/tmp/cowrie_exp04_arm_b_store.json")
)
ARM_B_RESTORE_RES = _run_phase(
    "with_cowrie", "restore", ARM_B_STATE_DIR, Path("/tmp/cowrie_exp04_arm_b_restore.json")
)
assert_connector_was_used(ARM_B_STORE_RES, ARM_B_RESTORE_RES, HMA_PROOF)

# Both arms passed their hard 64/64 greedy parity gates above (no receipt if either arm fails!).
# Record cross-arm exact match transparently (exact BF16 logit ties can break differently across cold engines).
_a_toks = ARM_A_RESTORE_RES["decode_token_ids"]
_b_toks = ARM_B_RESTORE_RES["decode_token_ids"]
_cross_diff = next((i for i, (x, y) in enumerate(zip(_a_toks, _b_toks)) if x != y), None)
print(
    f"\nFAIL-LOUD PAIRED A/B VERIFICATION PASSED: "
    f"ARM A parity=64/64, ARM B parity=64/64, SupportsHMA=True, "
    f"cross_arm_exact={_cross_diff is None} (first_diff={_cross_diff})."
)

In [ ]:
import json

SUMMARY = compute_ab_benchmark_summary(
    ARM_A_STORE_RES,
    ARM_A_RESTORE_RES,
    ARM_B_STORE_RES,
    ARM_B_RESTORE_RES,
    HMA_PROOF,
    GPU_INFO,
)
RECEIPT_PATH.write_text(json.dumps(SUMMARY, indent=2), encoding="utf-8")

ma = SUMMARY["arm_a_without_cowrie"]["metrics"]
mb = SUMMARY["arm_b_with_cowrie"]["metrics"]
ab = SUMMARY["ab"]
st_b = SUMMARY["arm_b_with_cowrie"]["connector_counters"]["phase_1_store"]["totals"]
rt_b = SUMMARY["arm_b_with_cowrie"]["connector_counters"]["phase_2_restore"]["totals"]
mf = GPU_INFO["memory_fit"]

print("=" * 88)
print("EXP-04 (CLOUD-LANE-01) — PAIRED A/B: COWRIE vs NATIVE vLLM PREFIX CACHE (ACROSS RESTART)")
print("=" * 88)
print(f"GPU / Compute Cap       : {GPU_INFO['gpu_name']} ({GPU_INFO['vram_total_gib']} GiB total, {GPU_INFO['vram_free_gib']} GiB free, sm_{GPU_INFO['compute_capability']})")
print(f"Model / Provenance      : {CONFIG['model_id']} [{CONFIG['provenance']}]")
print(f"Params / Weight Budget  : {CONFIG['params']:,} ({CONFIG['active_params']:,} active) | {CONFIG['weight_dtype']} ({mf['computed_weight_gib']:.2f} GiB raw, ~{mf['projected_kv_headroom_gib']:.2f} GiB KV headroom)")
print(f"vLLM / Cowrie / Config  : vLLM {HMA_PROOF['vllm_version']} | cowrie {HMA_PROOF['cowrie_git_sha']} | max_model_len={CONFIG['max_model_len']} | gpu_util={CONFIG['gpu_memory_utilization']} | seed={CONFIG['seed']} | eager={CONFIG['enforce_eager']}")
print(f"SupportsHMA (Runtime)   : {HMA_PROOF['issubclass_supports_hma']} (native={HMA_PROOF['supports_hma_native_import']}, live_all={rt_b['all_supports_hma']}) | layout={rt_b['kv_layouts']}")
print("-" * 88)
print(ab["arm_a_restart_summary"])
print("-" * 88)
print(f"{'Metric':<34} {'ARM A (without)':>16} {'ARM B (with)':>16} {'Delta (B-A)':>14} {'B vs A':>8}")
print("-" * 88)
for row in ab["delta_table"]:
    sp = f"{row['speedup_b_vs_a_x']:.3f}x" if row["speedup_b_vs_a_x"] is not None else "-"
    dl = f"{row['delta_b_minus_a']:+.2f}" if isinstance(row["delta_b_minus_a"], float) else (f"{row['delta_b_minus_a']:+d}" if isinstance(row["delta_b_minus_a"], int) else "-")
    print(f"{row['metric']:<34} {str(row['arm_a_without_cowrie']):>16} {str(row['arm_b_with_cowrie']):>16} {dl:>14} {sp:>8}")
print("-" * 88)
print(
    f"Greedy Parity           : ARM A={ma['greedy_token_parity']} ({ma['decode_output_tokens']}/{ma['decode_output_tokens']}) | "
    f"ARM B={mb['greedy_token_parity']} ({mb['decode_output_tokens']}/{mb['decode_output_tokens']}) | "
    f"Cross-Arm exact={ab['cross_arm_exact_token_match']} (prefix={ab['cross_arm_matching_prefix_tokens']}/{mb['decode_output_tokens']})"
)
print(
    f"HMAC & State Shards     : ARM A={ma['state_shards_count']} shards (hmac={ma['hmac_sha256_verified']}) | "
    f"ARM B={mb['state_shards_count']} shards, {mb['state_shards_total_bytes'] / (1024**2):.1f} MiB (hmac={mb['hmac_sha256_verified']})"
)
print(
    f"HEADLINE (Cowrie vs Native Post-Restart TTFT): "
    f"{ma['warm_ttft_ms']:.2f} ms -> {mb['warm_ttft_ms']:.2f} ms ({ab['warm_ttft_speedup_cowrie_vs_native_x']:.3f}x)"
)
print("-" * 88)
st_a = SUMMARY["arm_a_without_cowrie"]["connector_counters"]["phase_1_store"]["totals"]
rt_a = SUMMARY["arm_a_without_cowrie"]["connector_counters"]["phase_2_restore"]["totals"]
print("ARM A Connector Counters — Phase 1 (Store):", json.dumps(st_a, indent=2))
print("ARM A Connector Counters — Phase 2 (Restore):", json.dumps(rt_a, indent=2))
print("ARM B Connector Counters — Phase 1 (Store):", json.dumps(st_b, indent=2))
print("ARM B Connector Counters — Phase 2 (Restore):", json.dumps(rt_b, indent=2))
print("=" * 88)
print(f"VERDICT: {SUMMARY['verdict']}")
print(f"Saved paired A/B JSON receipt to: {RECEIPT_PATH}")

## 4. Colab Runtime Teardown (Prevent Idle CU Burn)

Download `/tmp/cowrie_exp04_receipt.json` into `receipts/` before unassigning the runtime. Set `EXP04_AUTO_UNASSIGN=1` (or flip `AUTO_UNASSIGN_COLAB_RUNTIME = True` in Cell 1) for unattended runs.

In [ ]:
if AUTO_UNASSIGN_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("Unassigning Colab runtime to stop CU burn...")
        runtime.unassign()
    except ImportError:
        print("Not running inside Google Colab; skipping runtime.unassign().")
else:
    print(
        f"Receipt ready at {RECEIPT_PATH}. "
        "Download it, then disconnect/delete the Colab runtime (or call google.colab.runtime.unassign()) "
        "to avoid burning idle Compute Units."
    )